### The core architectural difference between a GPU and a CPU:

A CPU is designed for sequential processing with a few powerful cores, while a GPU is built for massive parallel processing using thousands of smaller, simpler cores

Ref: https://www.rightnowai.co/guides/gpu-comparison/a100

In [ ]:
import torch
import pynvml #PyNVML (Python for NVIDIA Management Library) is a Python wrapper that allows you to monitor and manage NVIDIA GPUs programmatically

c:\Users\MRAZA3\work\repo_gitl_automation\loki-aud-closeloop\cuda_env\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
# Check if dedicated GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {torch.cuda.get_device_name(0)}")

Using device: NVIDIA RTX 1000 Ada Generation Laptop GPU


In [3]:
# A100 Memory
# Architecture: Ampere (GA100)

# Memory-efficient training for A100
torch.backends.cuda.matmul.allow_tf32 = True  # Enable TF32 for Ampere (GA100)
torch.backends.cudnn.allow_tf32 = True

# Difference between shared GPU and dedicated GPU memory:

Dedicated GPU memory is physical, high-speed RAM built directly onto your graphics card (often called VRAM). Shared GPU memory is essentially regular system RAM (your computer's main memory) that the operating system allows your graphics processor to use as an overflow buffer when the dedicated memory fills up.

The differences between the two dictate how smooth your system runs during demanding tasks like gaming, video editing, or 3D rendering.

![CPU-GPU memory sharing](data/img/cpu-gpu-shared-memory-dedicated-memory.png)

## Dedicated GPU Memory (VRAM)
What it is: Hardware built specifically for graphic-heavy tasks.
Speed: Extremely fast with massive memory bandwidth (often 20 to 100 times faster than system RAM).
Capacity: Ranges from 2 GB to 24 GB+ depending on your GPU.
Usage: Your GPU uses this first. It holds textures, models, and frame buffers so they render instantly.
Impact on Performance: Essential for gaming, AI processing, and heavy creative applications. Running out of this causes a severe drop in performance.

## Shared GPU Memory
What it is: A portion of your standard system RAM (DDR4 or DDR5) borrowed by the GPU.
Speed: Much slower than dedicated VRAM. Because it's shared, the GPU and CPU have to compete for the same bandwidth.
Capacity: Dynamically allocated. Windows typically assigns up to 50% of your total system RAM for this purpose.
Usage: Acts as an emergency backup. If a game or program demands more memory than your dedicated VRAM can hold, the system will spill over into this shared space to prevent a crash.
Impact on Performance: While it prevents system crashes, gaming or rendering on shared memory will cause severe stuttering and frame-rate drops due to the slow data-transfer speed across the PCIe bus.

Ref: https://irendering.net/understanding-the-dedicated-and-shared-gpu-memory/

In [4]:
# Check available dedicated NVIDIA GPU memory
pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)
info = pynvml.nvmlDeviceGetMemoryInfo(handle)
print(f"Free memory: {info.free / 1024**3:.1f} GB / 6 GB total")

Free memory: 5.8 GB / 6 GB total


In [5]:
# Recommended batch size calculation for A100
model_memory_gb = 2.0  # Adjust based on your model
batch_multiplier = (6 - model_memory_gb) / 1  # 1GB per batch unit
recommended_batch = int(batch_multiplier * 32)
print(f"Recommended batch size: {recommended_batch}")

Recommended batch size: 128
